# Tests

## Gedo monthly weather

This notebook tests the Copernicus client with a five-point polygon for the Gedo agropastoral area. It requests the longest configured monthly ERA5-Land range first and falls back to shorter ranges if CDS rejects the request because of cost/rate limits. Downloaded NetCDF and CSV outputs are cached in `project/data/weather`.


In [ ]:
from datetime import datetime
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'project').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project.clients.copernicus import (
    fetch_monthly_weather_for_polygon,
    normalize_polygon,
    polygon_bbox,
    summarize_weather,
)


In [ ]:
WEATHER_DIR = PROJECT_ROOT / 'project' / 'data' / 'weather'
WEATHER_DIR.mkdir(parents=True, exist_ok=True)

CURRENT_YEAR = datetime.now().year
TARGET_START_YEAR = CURRENT_YEAR - 20
TARGET_END_YEAR = CURRENT_YEAR - 1

FALLBACK_RANGES = [
    (TARGET_START_YEAR, TARGET_END_YEAR),
    (CURRENT_YEAR - 15, TARGET_END_YEAR),
    (CURRENT_YEAR - 10, TARGET_END_YEAR),
    (CURRENT_YEAR - 5, TARGET_END_YEAR),
]

GEDO_POLYGON = [
    # Coordinates are (longitude, latitude), converted from the DMS notes below.
    (42.8903167, 4.3140417),
    (43.0996111, 3.0833333),
    (41.5229722, 1.2651944),
    (40.9924083, 2.7923778),
    (42.0586444, 4.1903472),
]

print('Gedo points:', normalize_polygon(GEDO_POLYGON))
print('CDS bbox [north, west, south, east]:', polygon_bbox(GEDO_POLYGON, padding=0.1))
print('Target range:', TARGET_START_YEAR, TARGET_END_YEAR)


In [ ]:
def get_or_fetch_gedo_weather():
    errors = []

    for start_year, end_year in FALLBACK_RANGES:
        csv_path = WEATHER_DIR / f'gedo_monthly_weather_{start_year}_{end_year}.csv'
        if csv_path.ealsxists():
            df = pd.read_csv(csv_path, parse_dates=['month'])
            print(f'Loaded cached weather data: {csv_path}')
            return df, csv_path, (start_year, end_year), errors

        try:
            print(f'Requesting Gedo weather data: {start_year}-{end_year}')
            df = fetch_monthly_weather_for_polygon(
                polygon=GEDO_POLYGON,
                start_year=start_year,
                end_year=end_year,
                region_name='gedo',
                data_dir=WEATHER_DIR,
                grid=0.1,
            )
            df.to_csv(csv_path, index=False)
            print(f'Saved weather data: {csv_path}')
            return df, csv_path, (start_year, end_year), errors
        except Exception as exc:
            message = f'{start_year}-{end_year}: {type(exc).__name__}: {exc}'
            errors.append(message)
            lower_message = str(exc).lower()
            if 'cost' in lower_message or 'rate' in lower_message or 'too large' in lower_message:
                print('CDS rejected the request size; trying a shorter range.')
                continue
            raise

    raise RuntimeError('All configured weather ranges failed: ' + ' | '.join(errors))


gedo_weather, gedo_csv_path, used_range, fetch_errors = get_or_fetch_gedo_weather()
print('Used range:', used_range)
print('Rows:', len(gedo_weather))
fetch_errors


In [ ]:
gedo_weather.head(), gedo_weather.tail(), summarize_weather(gedo_weather)


In [ ]:
plot_df = gedo_weather.copy()
plot_df['month'] = pd.to_datetime(plot_df['month'])

metrics = [
    ('temperature_avg_c', 'Average temperature (C)'),
    ('relative_humidity_pct', 'Relative humidity (%)'),
    ('rainfall_mm_per_day', 'Rainfall (mm/day)'),
]

fig, axes = plt.subplots(len(metrics), 1, figsize=(14, 9), sharex=True)

for ax, (column, label) in zip(axes, metrics):
    ax.plot(plot_df['month'], plot_df[column], linewidth=1.5)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.25)

axes[-1].set_xlabel('Month')
fig.suptitle(f'Gedo monthly weather metrics ({used_range[0]}-{used_range[1]})', y=0.995)
fig.tight_layout()

plot_path = WEATHER_DIR / f'gedo_monthly_weather_{used_range[0]}_{used_range[1]}.png'
fig.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.show()

plot_path


### Original Gedo polygon notes

Gedo agropastoral:

- 4°18'50,55"N 42°53'25,14"E
- 3°05'00,00"N 43°05'58,60"E
- 1°15'54,70"N 41°31'22,70"E
- 2°47'32,56"N 40°59'32,67"E
- 4°11'25,25"N 42°03'31,12"E
